# Notebook 05 - Sistema Completo com Seguranca e Logging

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Integracao de todos os componentes
2. Sistema de seguranca e validacao
3. Logging detalhado para auditoria
4. Explainability das respostas
5. Demonstracao final do assistente

---
## 1. Configuracao do Ambiente

In [10]:
import os
import sys
import json
import logging
import torch
from datetime import datetime
from typing import TypedDict, List, Optional
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END

# Importar modulos centralizados
sys.path.append('..')
from src.model_loader import load_llm
from src.rag_module import (
    create_medical_documents,
    create_vector_store,
    create_retriever
)
from src.security import MedicalSecurityValidator, create_safety_report

load_dotenv()

# Configuracao do Modelo Fine-Tuned
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "../models/assistente_medico_final"

print("Sistema Assistente Medico - Versao Completa")
print("="*50)
print(f"Modelo base: {MODEL_NAME}")
print(f"Adapter: {ADAPTER_PATH}")

Sistema Assistente Medico - Versao Completa
Modelo base: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Adapter: ../models/assistente_medico_final


---
## 2. Sistema de Logging

Logging detalhado e essencial para auditoria e rastreabilidade em sistemas medicos.

In [11]:
# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('../logs/assistente_medico.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('AssistenteMedico')

def registrar_consulta(pergunta, resposta, fontes, confianca, medico_id="anonimo"):
    """Registra cada consulta para auditoria."""
    registro = {
        "timestamp": datetime.now().isoformat(),
        "medico_id": medico_id,
        "pergunta": pergunta[:200],  # Limitar para privacidade
        "resposta_gerada": resposta[:500],
        "fontes_consultadas": fontes,
        "nivel_confianca": confianca,
        "validacao_humana": False
    }
    
    logger.info("CONSULTA_ASSISTENTE", extra=registro)
    
    # Salvar em JSON para auditoria
    with open('../logs/auditoria.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(registro, ensure_ascii=False) + '\n')
    
    return registro

print("Sistema de logging configurado!")
print("Logs salvos em: ../logs/assistente_medico.log")
print("Auditoria em: ../logs/auditoria.jsonl")

Sistema de logging configurado!
Logs salvos em: ../logs/assistente_medico.log
Auditoria em: ../logs/auditoria.jsonl


---
## 3. Modulo de Seguranca

### 3.1 Regras de Seguranca

In [12]:
# Importar regras e funcoes de seguranca do modulo centralizado
from src.security import (
    MedicalSecurityValidator,
    SAFETY_PROMPT,
    PROHIBITED_WORDS,
    URGENCY_KEYWORDS,
    create_safety_report
)

# Regras de seguranca (alias para compatibilidade)
REGRAS_SEGURANCA = {
    "prompt_seguranca": SAFETY_PROMPT,
    "palavras_proibidas": PROHIBITED_WORDS,
    "alertas_urgencia": URGENCY_KEYWORDS
}

# Funcoes de validacao e urgencia (importadas do modulo centralizado)
def validar_resposta(resposta: str) -> dict:
    """Valida resposta usando MedicalSecurityValidator."""
    resultado = MedicalSecurityValidator.validate_response(resposta)
    return {
        "valida": resultado["is_safe"],
        "pontuacao_seguranca": resultado["safety_score"],
        "problemas": resultado["warnings"]
    }

def detectar_urgencia(pergunta: str) -> dict:
    """Detecta urgencia usando check_emergency_level."""
    resultado = MedicalSecurityValidator.check_emergency_level(pergunta)
    nivel = resultado["urgency_level"]
    return {
        "e_urgencia": nivel in ["ALTA", "MEDIA"],
        "nivel": nivel,
        "urgencias": resultado["recommendations"]
    }

print("Modulo de seguranca importado do modulo centralizado!")

Modulo de seguranca importado do modulo centralizado!


### 3.2 Funcao de Anonimizacao

In [13]:
# Importar funcao de anonimizacao do modulo centralizado
from src import anonymize_text

# Criar alias em portugues para compatibilidade
anonimizar_texto = anonymize_text

print("Funcao de anonimizacao importada do modulo centralizado!")

Funcao de anonimizacao importada do modulo centralizado!


---
## 4. Sistema Completo com LangGraph

In [14]:
from langchain_core.prompts import PromptTemplate

# FLAG: True = usar modelo fine-tuned, False = usar modelo base puro
USE_FINETUNED = True

# Carregar LLM usando modulo centralizado
llm = load_llm(
    model_name=MODEL_NAME,
    adapter_path=ADAPTER_PATH,
    use_finetuned=USE_FINETUNED
)

print(f"Modelo carregado com sucesso!")
print(f"Modo: {'Fine-tuned' if USE_FINETUNED else 'Base'}")

# Configurar RAG (Retrieval-Augmented Generation)
print("\nConfigurando RAG...")
vectorstore = create_vector_store()
retriever = create_retriever(vectorstore, search_k=3)
print(f"Vector store: {vectorstore.index.ntotal} vetores")
print("Retriever configurado!")

2026-09-14 01:47:37,635 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-14 01:47:37,646 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json?%2FTinyLlama%2FTinyLlama-1.1B-Chat-v1.0%2Fresolve%2Fmain%2Fconfig.json=&etag=%224ea05f1bc289d48ba9b92eea2f58ad8acd3dce5d%22 "HTTP/1.1 200 OK"
2026-09-14 01:47:37,825 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-14 01:47:37,836 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json?%2FTinyLlama%2FTinyLlama-1.1B-Chat-v1.0%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%22fa

Carregando modelo base...


2026-09-14 01:47:38,568 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-14 01:47:38,582 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json?%2FTinyLlama%2FTinyLlama-1.1B-Chat-v1.0%2Fresolve%2Fmain%2Fconfig.json=&etag=%224ea05f1bc289d48ba9b92eea2f58ad8acd3dce5d%22 "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 201/201 [00:03<00:00, 50.73it/s]
2026-09-14 01:47:42,778 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-14 01:47:42,788 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json?%2FTinyLlama%2FTinyLlam

Aplicando adapter LoRA...
Modelo carregado com sucesso!
Modo: Fine-tuned

Configurando RAG...
Documentos divididos em 8 chunks
Vector store criado com 8 vetores
Vector store: 8 vetores
Retriever configurado!


---
## 5. Construcao do Grafo Final

In [15]:
# ============================================
# ESTADO DO SISTEMA
# ============================================
class AssistenteState(TypedDict):
    pergunta: str
    pergunta_anonimizada: str
    classificacao: str
    nivel_urgencia: str
    documentos: List[Document]
    contexto: str
    resposta_bruta: str
    resposta_validada: str
    fontes: List[str]
    confianca: float
    pontuacao_seguranca: int
    alerta_urgencia: bool
    mensagem_alerta: str
    historico: List[str]
    timestamp: str

# ============================================
# NOS DO SISTEMA
# ============================================
def no_anonimizar(state: dict) -> dict:
    """Anonimiza a pergunta antes do processamento."""
    pergunta = state["pergunta"]
    pergunta_anon = anonimizar_texto(pergunta)
    historico = state.get("historico", [])
    historico.append("Pergunta anonimizada com sucesso")
    return {
        "pergunta_anonimizada": pergunta_anon,
        "historico": historico,
        "timestamp": datetime.now().isoformat()
    }

def no_verificar_urgencia(state: dict) -> dict:
    """Verifica se ha situacao de urgencia."""
    pergunta = state["pergunta"]
    resultado = detectar_urgencia(pergunta)
    historico = state.get("historico", [])
    if resultado["e_urgencia"]:
        historico.append(f"URGENCIA DETECTADA: {', '.join(resultado['urgencias'])}")
        return {
            "alerta_urgencia": True,
            "mensagem_alerta": f"ALERTA {resultado['nivel']}: Detectada possivel situacao de urgencia.",
            "nivel_urgencia": resultado["nivel"],
            "historico": historico
        }
    else:
        historico.append("Nenhuma urgencia detectada")
        return {
            "alerta_urgencia": False,
            "mensagem_alerta": "",
            "nivel_urgencia": "NORMAL",
            "historico": historico
        }

def no_classificar(state: dict) -> dict:
    """Classifica a consulta medica."""
    pergunta = state["pergunta_anonimizada"]
    prompt = PromptTemplate(
        template="""### Instruction:
Classifique: {pergunta}
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
""",
        input_variables=["pergunta"]
    )
    resposta = llm.invoke(prompt.format(pergunta=pergunta))
    classificacao = resposta.strip()
    historico = state.get("historico", [])
    historico.append(f"Classificacao: {classificacao}")
    return {
        "classificacao": classificacao,
        "historico": historico
    }

def no_recuperar(state: dict) -> dict:
    """Recupera documentos relevantes usando RAG real."""
    pergunta = state["pergunta_anonimizada"]
    docs = retriever.invoke(pergunta)
    historico = state.get("historico", [])
    historico.append(f"RAG: Recuperados {len(docs)} documentos relevantes")
    return {
        "documentos": docs,
        "historico": historico
    }

def no_gerar_resposta(state: dict) -> dict:
    """Gera a resposta com seguranca integrada."""
    pergunta = state["pergunta_anonimizada"]
    classificacao = state.get("classificacao", "")
    documentos = state.get("documentos", [])
    contexto = "\n".join([doc.page_content for doc in documentos])
    fontes = [doc.metadata.get("fonte", "") for doc in documentos]
    prompt = PromptTemplate(
        template="""### Instruction:
REGRAS INEGOCIAVEIS:
1. NUNCA prescreva medicamentos diretamente
2. NUNCA faca diagnosticos definitivos
3. SEMPRE inclua: 'Esta resposta e uma sugestao e deve ser validada por um medico'
4. SEMPRE cite a fonte do protocolo utilizado

Contexto: {contexto}
Pergunta: {pergunta}
Classificacao: {classificacao}

Responda de forma estruturada:
1. Analise clinica
2. Condutas recomendadas
3. Exames complementares
4. Fontes consultadas
5. AVISO DE SEGURANCA OBRIGATORIO

### Response:
""",
        input_variables=["contexto", "pergunta", "classificacao"]
    )
    resposta = llm.invoke(prompt.format(contexto=contexto, pergunta=pergunta, classificacao=classificacao))
    historico = state.get("historico", [])
    historico.append("Resposta gerada")
    return {
        "resposta_bruta": resposta,
        "fontes": fontes,
        "historico": historico
    }

def no_validar_seguranca(state: dict) -> dict:
    """Valida a resposta gerada contra as regras de seguranca."""
    resposta = state.get("resposta_bruta", "")
    validacao = MedicalSecurityValidator().validate_response(resposta)
    historico = state.get("historico", [])
    if validacao["is_safe"]:
        historico.append(f"Validacao APROVADA (pontuacao: {validacao['safety_score']})")
        return {"resposta_validada": resposta, "pontuacao_seguranca": validacao["safety_score"], "historico": historico}
    else:
        aviso = "\n\n[AVISO DE SEGURANCA] Esta resposta deve ser revisada por um medico especialista."
        historico.append(f"Validacao com OBSERVACOES")
        return {"resposta_validada": resposta + aviso, "pontuacao_seguranca": validacao["safety_score"], "historico": historico}

def no_registrar(state: dict) -> dict:
    """Registra a consulta para auditoria."""
    registrar_consulta(
        pergunta=state.get("pergunta", ""),
        resposta=state.get("resposta_validada", ""),
        fontes=state.get("fontes", []),
        confianca=state.get("confianca", 0)
    )
    historico = state.get("historico", [])
    historico.append("Consulta registrada para auditoria")
    return {"historico": historico}

print("Todos os nos do sistema implementados!")

Todos os nos do sistema implementados!


---
## 5. Construcao do Grafo Final

In [16]:
workflow = StateGraph(AssistenteState)

# Adicionar nos do grafo
workflow.add_node("anonimizar", no_anonimizar)
workflow.add_node("verificar_urgencia", no_verificar_urgencia)
workflow.add_node("classificar", no_classificar)
workflow.add_node("recuperar", no_recuperar)
workflow.add_node("gerar_resposta", no_gerar_resposta)
workflow.add_node("validar_seguranca", no_validar_seguranca)
workflow.add_node("registrar", no_registrar)

# Fluxo principal
workflow.set_entry_point("anonimizar")
workflow.add_edge("anonimizar", "verificar_urgencia")
workflow.add_edge("verificar_urgencia", "classificar")
workflow.add_edge("classificar", "recuperar")
workflow.add_edge("recuperar", "gerar_resposta")
workflow.add_edge("gerar_resposta", "validar_seguranca")
workflow.add_edge("validar_seguranca", "registrar")
workflow.add_edge("registrar", END)

# Compilar o grafo
assistente = workflow.compile()

print("Sistema Assistente Medico compilado!")
print("Fluxo: Anonimizar -> Urgencia -> Classificar -> RAG -> Resposta -> Seguranca -> Registrar -> Fim")

Sistema Assistente Medico compilado!
Fluxo: Anonimizar -> Urgencia -> Classificar -> RAG -> Resposta -> Seguranca -> Registrar -> Fim


---
## 6. Demonstracao do Sistema

In [17]:
# Funcao para executar o assistente
def executar_assistente(pergunta: str, medico_id: str = "anonimo") -> dict:
    """
    Executa o assistente medico completo.
    """
    print(f"\n{'='*60}")
    print("EXECUTANDO ASSISTENTE MEDICO")
    print("="*60)
    print(f"Pergunta: {pergunta}")
    print(f"Medico ID: {medico_id}")
    print("-"*60)
    
    entrada = {
        "pergunta": pergunta,
        "historico": []
    }
    
    resultado = assistente.invoke(entrada)
    
    # Exibir resultados
    print(f"\nClassificacao: {resultado['classificacao']}")
    print(f"Nivel urgencia: {resultado['nivel_urgencia']}")
    print(f"Pontuacao seguranca: {resultado['pontuacao_seguranca']}/100")
    print(f"Fontes: {resultado['fontes']}")
    
    if resultado['alerta_urgencia']:
        print(f"\n{'!'*60}")
        print(f"ALERTA: {resultado['mensagem_alerta']}")
        print(f"{'!'*60}")
    
    print(f"\nHistorico:")
    for i, h in enumerate(resultado['historico'], 1):
        print(f"  {i}. {h}")
    
    print(f"\n{'='*60}")
    print("RESPOSTA VALIDADA:")
    print("="*60)
    print(resultado['resposta_validada'])
    
    return resultado

print("Funcao de demonstracao pronta!")

Funcao de demonstracao pronta!


In [18]:
# DEMONSTRACAO 1: Consulta normal
resultado1 = executar_assistente(
    "Qual o protocolo para pneumonia hospitalar em pacientes idosos?",
    medico_id="dr_silva"
)


EXECUTANDO ASSISTENTE MEDICO
Pergunta: Qual o protocolo para pneumonia hospitalar em pacientes idosos?
Medico ID: dr_silva
------------------------------------------------------------


2026-09-14 01:48:14,461 - AssistenteMedico - INFO - CONSULTA_ASSISTENTE



Classificacao: ### Instruction:
Classifique: Qual o protocolo para pneumonia hospitalar em pacientes idosos?
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
Protocolo de protocolo de tratamento de pneumonia hospitalar em pacientes idosos
Nivel urgencia: NORMAL
Pontuacao seguranca: 100/100
Fontes: ['PROTO-001', 'PROTO-007', 'PROTO-002']

Historico:
  1. Pergunta anonimizada com sucesso
  2. Nenhuma urgencia detectada
  3. Classificacao: ### Instruction:
Classifique: Qual o protocolo para pneumonia hospitalar em pacientes idosos?
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
Protocolo de protocolo de tratamento de pneumonia hospitalar em pacientes idosos
  4. RAG: Recuperados 3 documentos relevantes
  5. Resposta gerada
  6. Validacao APROVADA (pontuacao: 100)
  7. Consulta registrada para auditoria

RESPOSTA VALIDADA:
### Instruction:
REG

In [19]:
# DEMONSTRACAO 2: Situacao de urgencia
resultado2 = executar_assistente(
    "Paciente com emergencia de infarto agudo do miocardio. Dor toracica intensa.",
    medico_id="dr_santos"
)


EXECUTANDO ASSISTENTE MEDICO
Pergunta: Paciente com emergencia de infarto agudo do miocardio. Dor toracica intensa.
Medico ID: dr_santos
------------------------------------------------------------


2026-09-14 01:48:34,321 - AssistenteMedico - INFO - CONSULTA_ASSISTENTE



Classificacao: ### Instruction:
Classifique: Paciente com emergencia de infarto agudo do miocardio. Dor toracica intensa.
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
Emergencia de infarto agudo do miocardio.
Nivel urgencia: ALTA
Pontuacao seguranca: 100/100
Fontes: ['PROTO-008', 'PROTO-003', 'PROTO-007']

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ALERTA: ALERTA ALTA: Detectada possivel situacao de urgencia.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Historico:
  1. Pergunta anonimizada com sucesso
  2. URGENCIA DETECTADA: PROCURE SOCORRO IMEDIATO (SAMU 192), Ligue para o hospital mais proximo
  3. Classificacao: ### Instruction:
Classifique: Paciente com emergencia de infarto agudo do miocardio. Dor toracica intensa.
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
Emergencia de infarto agudo do mio

In [20]:
# DEMONSTRACAO 3: Consulta com dados sensiveis (testar anonimizacao)
resultado3 = executar_assistente(
    "Paciente Maria Silva, CPF 123.456.789-00, com quadro de sepse. Qual conduta?",
    medico_id="dr橄榄" # Aqui ja passamos um ID, entao sera usado
)


EXECUTANDO ASSISTENTE MEDICO
Pergunta: Paciente Maria Silva, CPF 123.456.789-00, com quadro de sepse. Qual conduta?
Medico ID: dr橄榄
------------------------------------------------------------


2026-09-14 01:48:41,985 - AssistenteMedico - INFO - CONSULTA_ASSISTENTE



Classificacao: ### Instruction:
Classifique: Paciente [NOME-MASCARADO] Silva, CPF [CPF-MASCARADO], com quadro de sepse. Qual conduta?
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
CLINICA_GERAL
Nivel urgencia: NORMAL
Pontuacao seguranca: 100/100
Fontes: ['PROTO-006', 'PROTO-008', 'PROTO-003']

Historico:
  1. Pergunta anonimizada com sucesso
  2. Nenhuma urgencia detectada
  3. Classificacao: ### Instruction:
Classifique: Paciente [NOME-MASCARADO] Silva, CPF [CPF-MASCARADO], com quadro de sepse. Qual conduta?
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
CLINICA_GERAL
  4. RAG: Recuperados 3 documentos relevantes
  5. Resposta gerada
  6. Validacao APROVADA (pontuacao: 100)
  7. Consulta registrada para auditoria

RESPOSTA VALIDADA:
### Instruction:
REGRAS INEGOCIAVEIS:
1. NUNCA prescreva medicamentos diretamente
2. NUNCA faca diagnost

---
## 7. Explainability (Explicabilidade)

A explicabilidade e essencial em sistemas medicos para:
- Indicar fonte de cada informacao
- Mostrar nivel de confianca
- Registrar todas as etapas
- Permitir auditoria completa

In [21]:
def gerar_relatorio_explainability(resultado: dict) -> str:
    """
    Gera relatorio de explicabilidade da resposta.
    """
    relatorio = f"""
    RELATORIO DE EXPLICABILIDADE
    ============================
    
    Data/Hora: {resultado.get('timestamp', 'N/A')}
    
    1. CLASSIFICACAO:
       Categoria: {resultado.get('classificacao', 'N/A')}
       
    2. URGENCIA:
       Nivel: {resultado.get('nivel_urgencia', 'N/A')}
       Alerta: {'SIM' if resultado.get('alerta_urgencia') else 'NAO'}
       
    3. FONTES CONSULTADAS:
       {chr(10).join(f'- {f}' for f in resultado.get('fontes', []))}
    
    4. SEGURANCA:
       Pontuacao: {resultado.get('pontuacao_seguranca', 0)}/100
       
    5. HISTORICO DE DECISOES:
       {chr(10).join(f'   {i}. {h}' for i, h in enumerate(resultado.get('historico', []), 1))}
    
    6. AVISO LEGAL:
       Esta resposta foi gerada por Inteligencia Artificial.
       Todas as informacoes devem ser validadas por um medico
       especialista antes da decisao clinica.
    
    7. CONFORMIDADE:
       - LGPD: Dados anonimizados
       - CFF: Respeita etica medica
       - ANVISA: Seguranca de software medico
    """
    
    return relatorio

# Gerar relatorio para a ultima consulta
if resultado3:
    print(gerar_relatorio_explainability(resultado3))


    RELATORIO DE EXPLICABILIDADE

    Data/Hora: 2026-09-14T01:48:34.334131

    1. CLASSIFICACAO:
       Categoria: ### Instruction:
Classifique: Paciente [NOME-MASCARADO] Silva, CPF [CPF-MASCARADO], com quadro de sepse. Qual conduta?
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
CLINICA_GERAL

    2. URGENCIA:
       Nivel: NORMAL
       Alerta: NAO

    3. FONTES CONSULTADAS:
       - PROTO-006
- PROTO-008
- PROTO-003

    4. SEGURANCA:
       Pontuacao: 100/100

    5. HISTORICO DE DECISOES:
          1. Pergunta anonimizada com sucesso
   2. Nenhuma urgencia detectada
   3. Classificacao: ### Instruction:
Classifique: Paciente [NOME-MASCARADO] Silva, CPF [CPF-MASCARADO], com quadro de sepse. Qual conduta?
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
CLINICA_GERAL
   4. RAG: Recuperados 3 documentos relevantes
   5. Resposta gerad

---
## 8. Metricas e Monitoramento

In [22]:
# Carregar e analisar logs de auditoria
import os

log_file = '../logs/auditoria.jsonl'

if os.path.exists(log_file):
    registros = []
    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                registros.append(json.loads(line))
    
    print(f"Total de consultas registradas: {len(registros)}")
    
    # Analise basica
    if registros:
        confiancas = [r.get('nivel_confianca', 0) for r in registros]
        print(f"Confianca media: {sum(confiancas)/len(confiancas):.2%}")
        print(f"\nUltimas consultas:")
        for r in registros[-3:]:
            print(f"  - {r.get('timestamp', 'N/A')}: {r.get('pergunta', 'N/A')[:50]}...")
else:
    print("Nenhum log de auditoria encontrado ainda.")
    print("Execute algumas consultas primeiro.")

Total de consultas registradas: 22
Confianca media: 0.00%

Ultimas consultas:
  - 2026-09-14T01:48:14.461056: Qual o protocolo para pneumonia hospitalar em paci...
  - 2026-09-14T01:48:34.320991: Paciente com emergencia de infarto agudo do miocar...
  - 2026-09-14T01:48:41.985930: Paciente Maria Silva, CPF 123.456.789-00, com quad...


---
## 9. Resumo Final

### Sistema Implementado:

| Componente | Descricao |
|------------|----------|
| **Fine-Tuning** | Modelo treinado com dados medicos (Notebook 02) |
| **LangChain** | Prompts, Chains, Loaders, Agents (Notebook 03) |
| **LangGraph** | Grafo de decisao com fluxos condicionais (Notebook 04) |
| **RAG** | Recuperacao de documentos + Geracao (Notebook 04) |
| **Seguranca** | Validacao, anonimizacao, regras medicas |
| **Logging** | Auditoria detalhada de todas as consultas |
| **Explainability** | Transparencia total nas decisoes |

### Fluxo Completo:
```
Pergunta -> Anonimizacao -> Verificacao Urgencia -> Classificacao ->
Recuperacao (RAG) -> Geracao -> Validacao Seguranca -> Registro -> Resposta
```



In [23]:
print("\n" + "="*60)
print("ASSISTENTE MEDICO - SISTEMA COMPLETO")
print("="*60)
print("Tech Challenge Fase 3 - FIAP")
print("Projeto: Assistente Virtual Medico")
print("Tecnologias: LangChain, LangGraph, Fine-Tuning, RAG")
print("="*60)
print("\nSistema pronto para demonstracao!")
print("Execute as celulas de demonstracao para testar o assistente.")


ASSISTENTE MEDICO - SISTEMA COMPLETO
Tech Challenge Fase 3 - FIAP
Projeto: Assistente Virtual Medico
Tecnologias: LangChain, LangGraph, Fine-Tuning, RAG

Sistema pronto para demonstracao!
Execute as celulas de demonstracao para testar o assistente.
